# TN1 — Kiến trúc: TCN so với DS-TCN

Đổi **đúng một biến** so với TN0: thay LSTM của MobiVital bằng TCN. Mọi thứ khác giữ nguyên cấu hình tác giả công bố — 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr_threshold` 0.9, không RevIN (để dành TN2).

## Hai câu hỏi

| nhóm | câu hỏi | cấu hình |
|---|---|---|
| **nhẹ** | Thu nhỏ 90–96% thì còn tốt hơn LSTM không? | LSTM · TCN-64 · DS-TCN-64 |
| **ngang tham số** | Cho lượng tham số gần bằng nhau thì kiến trúc nào hơn? | kiến trúc thắng nhóm nhẹ, nâng kênh ngang LSTM |

## Các cấu hình

| model | kênh | tham số | so LSTM |
|---|---|---|---|
| LSTM (MobiVital) | hidden 352 | 1.502.713 | — |
| TCN | 64 | 151.513 | −90% |
| DS-TCN | 64 | 56.281 | −96% |
| TCN | 200 | 1.452.625 | −3% |
| DS-TCN | 352 | 1.525.945 | +2% |

Chạy **4 cấu hình**: ba cấu hình nhóm nhẹ, rồi một cấu hình ngang tham số dùng kiến trúc thắng. Mỗi cấu hình 4 fold, một seed.

## Giao thức

Bốn fold cố định trên tám người `A B C D E F K L`, dùng y nguyên cho mọi thí nghiệm:

```
val_AB   train C D E F K L    chấm A B
val_CE   train A B D F K L    chấm C E
val_DF   train A B C E K L    chấm D F
val_KL   train A B C D E F    chấm K L
```

**`G H I J` không được đụng tới.** Chúng chỉ dùng ở bước công bố cuối, sau khi mọi cấu hình đã chốt — xem `docs/PROTOCOL.md`.

Điểm chấm trên **buổi ghi thô**, model tự chọn kênh, không nhìn nhịp thở thật. Cửa sổ train thì có lọc bằng `corr > 0.9`, nên không được dùng cửa sổ để chấm.

## Cơ sở chọn tham số kiến trúc

`kernel=3`, `n_blocks=6`, hai tầng conv mỗi khối, `dropout=0.0` — mỗi số đều trích dẫn được, xem [`docs/THAM_CHIEU.md`](../docs/THAM_CHIEU.md). Ràng buộc quan trọng nhất: **tầm nhìn phải phủ hết 200 mẫu vào**; `k=3, n=6` cho 253 mẫu.

Kết quả vào `runs/tn1/`, cuối notebook nén thành `runs/tn1.zip`.


## 1. Chuẩn bị Colab


Mount Drive để lấy lại cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Tải mã nguồn rồi vào thư mục đó. `setup_colab.py` clone MobiVital và ghim commit `4319731d` — `src/mobivital_reference.py` mượn sáu hàm từ repo họ.


In [ ]:
# Phải clone repo trước, vì setup_colab.py nằm bên trong chính repo đó.
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py


Lấy `by_user/` và `windows/` từ Drive. TN1 **không cần CSV thô 13 GB** — chỉ train trên cửa sổ đã cắt và chấm trên `by_user/*.npz`.


In [ ]:
!python scripts/restore_processed_data_on_drive.py


## 2. Đo tốc độ trước khi chạy

Chạy hết TN1 mất vài giờ. Đo trước một batch của từng kiến trúc để ước đúng thời gian, và để trả lời câu "TCN nhẹ hơn thì có nhanh hơn không".

Đo trên dữ liệu ngẫu nhiên đúng kích thước thật — tốc độ chỉ phụ thuộc hình dạng tensor.


In [ ]:
!python scripts/do_toc_do.py


## 3. Nhóm nhẹ — ba cấu hình

Mỗi lệnh chạy trọn 4 fold rồi ghi 5 dòng vào `runs/summary.csv`: bốn dòng fold và một dòng `TONG` mang `cv_score` cùng `cv_std`.


**LSTM** — mốc so sánh. Phải chạy trong CV, vì số LSTM ở TN0 đo trên `G H I J` chứ không trên fold.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model lstm


**TCN-64** — tích chập nhân quả giãn dần, 151.513 tham số, nhẹ hơn LSTM 90%.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model tcn --channels 64


**DS-TCN-64** — tách depthwise và pointwise, 56.281 tham số, nhẹ hơn LSTM 96%.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 64


Kết quả nhóm nhẹ:


In [ ]:
!python scripts/compare_cv.py --experiment tn1 --so-doi lstm


## 4. Nhóm ngang tham số — một cấu hình

Chạy kiến trúc **thắng ở mục 3**, nâng số kênh cho tham số xấp xỉ LSTM. Câu hỏi khác hẳn nhóm nhẹ: bỏ ràng buộc kích thước thì kiến trúc nào dự báo tốt hơn.

Bai et al. mục A.1 chính là làm thế này — *"the number of hidden units was chosen so that the model size is approximately at the same level as the recurrent models with which we are comparing"*.

Chạy **một** trong hai ô dưới, tuỳ kiến trúc nào thắng:

| thắng ở mục 3 | chạy ô | tham số |
|---|---|---|
| TCN | `--model tcn --channels 200` | 1.452.625 |
| DS-TCN | `--model ds_tcn --channels 352` | 1.525.945 |


In [ ]:
# chạy ô này nếu TCN thắng ở mục 3
!python scripts/run_cv.py --experiment tn1 --model tcn --channels 200


In [ ]:
# chạy ô này nếu DS-TCN thắng ở mục 3
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 352


## 5. So sánh

Hai bảng:

1. **cv_score** — điểm trung bình 4 fold của từng cấu hình, kèm chi tiết từng fold và độ lệch chuẩn
2. **thắng / hoà / thua** — so **từng** buổi ghi với LSTM, trên đủ 1289 buổi của tám người dev

Bảng 2 cần thiết vì hai cấu hình chênh nhau 0.003 điểm trung bình có thể là tốt hơn đều khắp, hoặc thắng đậm vài buổi mà thua nhẹ phần lớn. Trung bình không phân biệt được.


In [ ]:
!python scripts/compare_cv.py --experiment tn1 --so-doi lstm


## 6. Lưu kết quả

Nén cả thư mục `runs/tn1/` thành `runs/tn1.zip`, và chép sang Drive. Giải nén lại bằng `unzip tn1.zip -d runs/`.


In [ ]:
!python scripts/save_results.py tn1
